# Gather all variables for the database

Loop the physics pipeline from notebook 01 over all 9,700 instances and produce one row per instance. The output is `dataset.csv`, the database used by EDA and ML.

**Variables collected per instance:**

*Identifiers and labels (the ML targets):*
- `instance_id` -- row index in the hits file (unique key)
- `event_id` -- 0–99 (template id, ~97 instances per template)
- `is_cc` -- True if any charged-lepton primary present
- `nu_pdg` -- PDG of the incoming neutrino
- `nu_flavour` -- 'nue' / 'numu' / 'nutau'

*Truth quantities:*
- `nu_energy_truth`, `nu_dir_truth_{x,y,z}`, `zenith_truth_deg`, `L_truth_km`

*Reconstructed quantities:*
- `nu_energy_reco`, `nu_dir_reco_{x,y,z}`, `zenith_reco_deg`, `L_reco_km`

*Topology features:*
- `n_primaries`, `n_total_particles`
- `n_primary_leptons`, `n_primary_protons`, `n_primary_pions`, `n_primary_photons`, `n_primary_other`
- `leading_lepton_pdg`, `leading_lepton_energy`

*Detector observables (no MC truth):*
- `n_hits`, `n_hits_w`, `n_hits_v`, `n_hits_u`
- `total_adc`, `mean_adc`, `max_adc`
- `drift_range`, `drift_std`
- `channel_range_w`, `channel_range_u`, `channel_range_v`
- `hit_density`

*Reconstruction quality:*
- `n_points_raw`, `n_points_filtered`
- `direction_recon_ok`, `is_ccqe_topology`, `is_ccqe_reconstructed`

## 1. Imports & constants (mirror of notebook 01)

In [1]:
import numpy as np
import pandas as pd
import uproot
from sklearn.decomposition import PCA
from tqdm.auto import tqdm

# ----- Constants (subset relevant to dataset build) -----
NEUTRINO_PDGS       = {12, -12, 14, -14, 16, -16}
CHARGED_LEPTON_PDGS = {11, -11, 13, -13, 15, -15}
PROTON_PDG          = 2212
PION_PDGS           = {211, -211, 111}
PHOTON_PDG          = 22
SHOWER_PDGS         = {11, -11, 22}

PDG_NAME = {11:'e-', -11:'e+', 12:'nu_e', -12:'nu_e_bar', 13:'mu-', -13:'mu+',
            14:'nu_mu', -14:'nu_mu_bar', 15:'tau-', -15:'tau+',
            16:'nu_tau', -16:'nu_tau_bar', 22:'gamma',
            211:'pi+', -211:'pi-', 2212:'p', -2212:'pbar', 2112:'n', -2112:'nbar'}

PARTICLE_MASSES = {11:0.000511, 13:0.105658, 15:1.77686, 22:0.0,
                   211:0.13957, 2212:0.938272, 2112:0.939565}

# Wire-plane angles
THETA_U, THETA_V, THETA_W = +0.623257100582, -0.623257100582, 0.0
COS_U, SIN_U = np.cos(THETA_U), np.sin(THETA_U)
COS_V, SIN_V = np.cos(THETA_V), np.sin(THETA_V)
COS_W, SIN_W = np.cos(THETA_W), np.sin(THETA_W)
SIN_DVU = np.sin(THETA_V - THETA_U)
SIN_DWV = np.sin(THETA_W - THETA_V)
SIN_DUW = np.sin(THETA_U - THETA_W)

EARTH_RADIUS_KM, ATMOS_PROD_HEIGHT_KM, DETECTOR_DEPTH_KM = 6371.0, 15.0, 1.5
DRIFT_BIN_WIDTH_CM, LINE_FIT_DIST_CM = 0.5, 2.0
MIN_POINTS_FOR_LINE, MIN_HITS_DIRECTION = 3, 3
MAX_RAW_3D_POINTS = 15_000

def uw_to_y(u, w): return (w*COS_U - u*COS_W) / SIN_DUW
def uw_to_z(u, w): return (w*SIN_U - u*SIN_W) / SIN_DUW
def vw_to_y(v, w): return (v*COS_W - w*COS_V) / SIN_DWV
def vw_to_z(v, w): return (v*SIN_W - w*SIN_V) / SIN_DWV
def yz_to_u(y, z): return z*COS_U - y*SIN_U
def yz_to_v(y, z): return z*COS_V - y*SIN_V

## 2. Load ROOT files and build per-instance MC index

In [2]:
MC_PATH, HITS_PATH = 'mc_0.root', 'hits_0.root'

mc_branches  = ['event_id','mc_id','pdg','energy','visible_energy',
                'mom_vec','vtx_vec','parent_id','n_hits_total']
hit_branches = ['event_id','plane','mc_id','drift','channel','adc','width']

def load_root(path, tree, branches):
    out = {}
    with uproot.open(f'{path}:{tree}') as t:
        for b in branches:
            out[b] = t[b].array(library='np')
    return out

print('Loading MC...');   mc   = load_root(MC_PATH,   'mc',   mc_branches)
print('Loading hits...'); hits = load_root(HITS_PATH, 'hits', hit_branches)
n_instances = len(hits['event_id'])

# Each MC instance starts at parent_id == -1
instance_starts = np.where(mc['parent_id'] == -1)[0]
instance_ends   = np.append(instance_starts[1:], len(mc['parent_id']))
assert len(instance_starts) == n_instances, 'MC instances vs hits rows mismatch'
print(f'Loaded {n_instances:,} instances')

Loading MC...


Loading hits...


Loaded 9,700 instances


## 3. Physics primitives (copied from notebook 01)

Same functions as `01_physics.ipynb`, repeated here so this notebook is self-contained. If you change one, change both.

In [3]:
def group_drifts(drift_array, tolerance=DRIFT_BIN_WIDTH_CM):
    if len(drift_array) == 0: return []
    sorted_d = np.sort(drift_array)
    groups = [[float(sorted_d[0])]]
    for d in sorted_d[1:]:
        if d - groups[-1][0] <= tolerance: groups[-1].append(float(d))
        else:                              groups.append([float(d)])
    return groups

def reconstruct_space_points(plane, drift, adc, channel, mc_id):
    groups = group_drifts(drift)
    coords, adc_out, mc_out, res_out = [], [], [], []
    for bin_d in groups:
        x_avg = float(np.mean(bin_d))
        bin_mask = (drift >= bin_d[0]) & (drift <= bin_d[-1])
        for mid in np.unique(mc_id[bin_mask]):
            sel = bin_mask & (mc_id == mid)
            u_idx = np.where(sel & (plane == 2))[0]
            v_idx = np.where(sel & (plane == 1))[0]
            w_idx = np.where(sel & (plane == 0))[0]
            if len(w_idx) == 0: continue
            for wi in w_idx:
                wc, wa = channel[wi], adc[wi]
                if len(u_idx) > 0 and len(v_idx) > 0:
                    for ui in u_idx:
                        uc = channel[ui]; y, z = uw_to_y(uc,wc), uw_to_z(uc,wc)
                        v_res = min(abs(yz_to_v(y,z) - channel[vh]) for vh in v_idx)
                        coords.append((x_avg,y,z)); adc_out.append(wa)
                        mc_out.append(int(mid)); res_out.append((np.nan, v_res))
                    for vi in v_idx:
                        vc = channel[vi]; y, z = vw_to_y(vc,wc), vw_to_z(vc,wc)
                        u_res = min(abs(yz_to_u(y,z) - channel[uh]) for uh in u_idx)
                        coords.append((x_avg,y,z)); adc_out.append(wa)
                        mc_out.append(int(mid)); res_out.append((u_res, np.nan))
                elif len(u_idx) > 0:
                    for ui in u_idx:
                        uc = channel[ui]
                        coords.append((x_avg, uw_to_y(uc,wc), uw_to_z(uc,wc)))
                        adc_out.append(wa); mc_out.append(int(mid)); res_out.append((np.nan,np.nan))
                elif len(v_idx) > 0:
                    for vi in v_idx:
                        vc = channel[vi]
                        coords.append((x_avg, vw_to_y(vc,wc), vw_to_z(vc,wc)))
                        adc_out.append(wa); mc_out.append(int(mid)); res_out.append((np.nan,np.nan))
    return (np.asarray(coords).reshape(-1,3),
            np.asarray(adc_out), np.asarray(mc_out, dtype=int),
            np.asarray(res_out).reshape(-1,2))

def fit_3d_line(points):
    if len(points) < 2: return None, None
    centroid = np.mean(points, axis=0)
    _, _, vh = np.linalg.svd(points - centroid, full_matrices=False)
    return centroid, vh[0]

def point_to_line_distance(p, lp, ld):
    vec = p - lp
    return float(np.linalg.norm(vec - np.dot(vec, ld) * ld))

def filter_space_points(coords, adc, mc_ids, residuals,
                        distance_threshold=LINE_FIT_DIST_CM, min_points=MIN_POINTS_FOR_LINE):
    n = len(coords)
    if n == 0: return coords, adc, mc_ids, residuals
    flat = residuals.flatten(); valid = flat[~np.isnan(flat)]
    rms = float(np.sqrt(np.mean(valid**2))) if len(valid) > 0 else 1.0
    good = np.zeros(n, dtype=bool); no_res = np.zeros(n, dtype=bool)
    for i in range(n):
        u, v = residuals[i]
        hu, hv = not np.isnan(u), not np.isnan(v)
        if hu and hv:   good[i] = (u < rms) and (v < rms)
        elif hu:        good[i] = u < rms
        elif hv:        good[i] = v < rms
        else:           no_res[i] = True
    fitted = {}; pdist = np.full(n, np.nan)
    for mid in np.unique(mc_ids):
        m = mc_ids == mid
        trusted = coords[m & good]
        if len(trusted) >= min_points:
            lp, ld = fit_3d_line(trusted); fitted[int(mid)] = (lp, ld)
            for i in np.where(m)[0]: pdist[i] = point_to_line_distance(coords[i], lp, ld)
    final = good.copy()
    for i in np.where(no_res)[0]:
        mid = int(mc_ids[i])
        final[i] = (pdist[i] < distance_threshold) if mid in fitted else True
    return coords[final], adc[final], mc_ids[final], residuals[final]

def reconstruct_directions(primaries, coords_f, mc_ids_f):
    out = {}
    for p in primaries:
        if not p['has_hits']: continue
        mid = p['mc_id']; pdg_abs = abs(p['pdg'])
        track = coords_f[mc_ids_f == mid]
        if len(track) < MIN_HITS_DIRECTION: continue
        vertex = np.array([p['vtx_x'], p['vtx_y'], p['vtx_z']])
        centred = track - vertex
        if pdg_abs in SHOWER_PDGS:
            norms = np.linalg.norm(centred, axis=1, keepdims=True)
            unit = np.divide(centred, norms, out=np.zeros_like(centred), where=norms > 0)
            w = norms.squeeze() / norms.sum()
            d = np.average(unit, axis=0, weights=w)
            n = np.linalg.norm(d)
            if n == 0: continue
            d /= n
        else:
            pca = PCA(n_components=3); pca.fit(centred)
            d = pca.components_[0]
        furthest = centred[np.argmax(np.linalg.norm(centred, axis=1))]
        if np.dot(d, furthest) < 0: d = -d
        out[mid] = {'direction': d, 'pdg': p['pdg']}
    return out

def reconstruct_neutrino(primaries, directions, parent_nu):
    if parent_nu is None: return None
    p_total = np.zeros(3); n_done = 0
    for p in primaries:
        if p['mc_id'] not in directions or not p['has_hits']: continue
        mass = PARTICLE_MASSES.get(abs(p['pdg']), 0.0)
        E = p['energy']; pmag = float(np.sqrt(max(E*E - mass*mass, 0.0)))
        p_total += pmag * directions[p['mc_id']]['direction']
        n_done += 1
    pmag_total = float(np.linalg.norm(p_total))
    direction = p_total / pmag_total if pmag_total > 0 else np.zeros(3)
    return {'momentum_mag': pmag_total, 'direction': direction, 'energy': pmag_total,
            'n_reconstructed': n_done}

def zenith_angle_deg(direction):
    n = np.linalg.norm(direction)
    if n == 0: return float('nan')
    return float(np.degrees(np.arccos(max(-1.0, min(1.0, -float(direction[1])/n)))))

def baseline_km(zenith_rad,
                production_height=ATMOS_PROD_HEIGHT_KM,
                detector_depth=DETECTOR_DEPTH_KM,
                R=EARTH_RADIUS_KM):
    if not np.isfinite(zenith_rad): return float('nan')
    a = R - detector_depth; b = R + production_height
    cos_z = np.cos(zenith_rad)
    disc = a*a*cos_z*cos_z + (b*b - a*a)
    if disc < 0: return float('nan')
    return float(-a*cos_z + np.sqrt(disc))

## 4. Per-instance feature extraction

In [4]:
def build_particles(mc_ev, hit_mc_ids):
    """List of particle dicts for one instance."""
    hit_set = set(np.unique(hit_mc_ids).tolist())
    pdg2pdg = dict(zip(mc_ev['mc_id'].tolist(), mc_ev['pdg'].tolist()))
    out = []
    for i in range(len(mc_ev['mc_id'])):
        parent_id = int(mc_ev['parent_id'][i])
        parent_pdg = int(pdg2pdg.get(parent_id, 0)) if parent_id != -1 else None
        mom = np.asarray(mc_ev['mom_vec'][i], dtype=float)
        vtx = np.asarray(mc_ev['vtx_vec'][i], dtype=float)
        out.append({
            'mc_id': int(mc_ev['mc_id'][i]), 'pdg': int(mc_ev['pdg'][i]),
            'name': PDG_NAME.get(int(mc_ev['pdg'][i]), f"PDG_{int(mc_ev['pdg'][i])}"),
            'energy': float(mc_ev['energy'][i]),
            'mom_x': float(mom[0]), 'mom_y': float(mom[1]), 'mom_z': float(mom[2]),
            'momentum_mag': float(np.linalg.norm(mom)),
            'vtx_x': float(vtx[0]), 'vtx_y': float(vtx[1]), 'vtx_z': float(vtx[2]),
            'parent_id': parent_id, 'parent_pdg': parent_pdg,
            'has_hits': int(mc_ev['mc_id'][i]) in hit_set,
        })
    return out

def find_parent_neutrino(particles):
    for p in particles:
        if p['parent_id'] == -1 and abs(p['pdg']) in NEUTRINO_PDGS:
            return p
    return None

def find_primaries(particles, parent_nu):
    if parent_nu is None: return []
    return [p for p in particles if p['parent_id'] == parent_nu['mc_id']]

def detector_observables(plane, drift, adc, channel):
    """Hit-level summary statistics. NO MC truth used."""
    out = {
        'n_hits':   int(len(plane)),
        'n_hits_w': int(np.sum(plane == 0)),
        'n_hits_v': int(np.sum(plane == 1)),
        'n_hits_u': int(np.sum(plane == 2)),
    }
    if len(adc) > 0:
        out.update({
            'total_adc': float(np.sum(adc)),
            'mean_adc':  float(np.mean(adc)),
            'max_adc':   float(np.max(adc)),
            'drift_range': float(np.ptp(drift)),
            'drift_std':   float(np.std(drift)),
        })
        for pid, suf in [(0, 'w'), (1, 'v'), (2, 'u')]:
            ch = channel[plane == pid]
            out[f'channel_range_{suf}'] = float(np.ptp(ch)) if len(ch) > 0 else 0.0
        out['hit_density'] = (out['n_hits'] / out['drift_range']
                              if out['drift_range'] > 0 else 0.0)
    else:
        for k in ['total_adc','mean_adc','max_adc','drift_range','drift_std',
                  'channel_range_w','channel_range_u','channel_range_v','hit_density']:
            out[k] = 0.0
    return out

FLAVOUR_MAP = {12:'nue', -12:'nue', 14:'numu', -14:'numu', 16:'nutau', -16:'nutau'}

def process_instance(instance_id):
    """Run the full pipeline for one instance and return one row dict."""
    # ----- Hits -----
    plane    = hits['plane'][instance_id]
    drift    = hits['drift'][instance_id]
    adc      = hits['adc'][instance_id]
    channel  = hits['channel'][instance_id]
    hit_mc   = hits['mc_id'][instance_id]
    
    # ----- MC slice for this instance -----
    s, e = instance_starts[instance_id], instance_ends[instance_id]
    mc_ev = {k: v[s:e] for k, v in mc.items()}
    
    particles = build_particles(mc_ev, hit_mc)
    parent_nu = find_parent_neutrino(particles)
    primaries = find_primaries(particles, parent_nu)
    
    # ----- Labels -----
    primary_pdgs = [p['pdg'] for p in primaries]
    is_cc = any(abs(pdg) in CHARGED_LEPTON_PDGS for pdg in primary_pdgs)
    nu_pdg = parent_nu['pdg'] if parent_nu else None
    nu_flavour = FLAVOUR_MAP.get(nu_pdg, 'unknown') if nu_pdg is not None else 'unknown'
    
    # ----- Topology counts -----
    n_lep = sum(1 for pdg in primary_pdgs if abs(pdg) in CHARGED_LEPTON_PDGS)
    n_pro = sum(1 for pdg in primary_pdgs if pdg == PROTON_PDG)
    n_pi  = sum(1 for pdg in primary_pdgs if pdg in PION_PDGS)
    n_g   = sum(1 for pdg in primary_pdgs if pdg == PHOTON_PDG)
    n_other = len(primary_pdgs) - n_lep - n_pro - n_pi - n_g
    
    leading_lep = next((p for p in primaries if abs(p['pdg']) in CHARGED_LEPTON_PDGS), None)
    
    # ----- Truth kinematics -----
    if parent_nu is not None:
        nu_truth_dir = np.array([parent_nu['mom_x'], parent_nu['mom_y'], parent_nu['mom_z']])
        n = np.linalg.norm(nu_truth_dir)
        if n > 0: nu_truth_dir = nu_truth_dir / n
        zen_t = zenith_angle_deg(nu_truth_dir)
        L_t   = baseline_km(np.radians(zen_t)) if np.isfinite(zen_t) else float('nan')
        nu_e_truth = parent_nu['energy']
    else:
        nu_truth_dir = np.full(3, np.nan); zen_t = float('nan'); L_t = float('nan'); nu_e_truth = float('nan')
    
    # ----- Detector observables -----
    obs = detector_observables(plane, drift, adc, channel)
    
    # ----- 3D reconstruction -----
    n_raw, n_filt = 0, 0
    nu_e_reco, zen_r, L_r = float('nan'), float('nan'), float('nan')
    nu_dir_reco = np.full(3, np.nan)
    direction_ok = False
    
    if obs['n_hits'] > 0:
        coords, adc3d, mc3d, res = reconstruct_space_points(plane, drift, adc, channel, hit_mc)
        n_raw = len(coords)
        if 0 < n_raw <= MAX_RAW_3D_POINTS:
            coords_f, adc_f, mc3d_f, res_f = filter_space_points(coords, adc3d, mc3d, res)
            n_filt = len(coords_f)
            if n_filt > 0:
                directions = reconstruct_directions(primaries, coords_f, mc3d_f)
                nu = reconstruct_neutrino(primaries, directions, parent_nu)
                if nu is not None and nu['momentum_mag'] > 0:
                    nu_e_reco = nu['energy']
                    nu_dir_reco = nu['direction']
                    zen_r = zenith_angle_deg(nu_dir_reco)
                    L_r = baseline_km(np.radians(zen_r)) if np.isfinite(zen_r) else float('nan')
                    direction_ok = (nu['n_reconstructed'] == sum(1 for p in primaries if p['has_hits']))
    
    # ----- CCQE flag (legacy) -----
    is_ccqe_topology = (n_lep == 1 and n_pro == 1 and len(primaries) == 2)
    is_ccqe_reco     = is_ccqe_topology and direction_ok
    
    # ----- Pack the row -----
    row = {
        'instance_id': instance_id,
        'event_id':    int(hits['event_id'][instance_id]),
        'is_cc':       is_cc,
        'nu_pdg':      nu_pdg,
        'nu_flavour':  nu_flavour,
        # Truth
        'nu_energy_truth':  nu_e_truth,
        'nu_dir_truth_x':   float(nu_truth_dir[0]),
        'nu_dir_truth_y':   float(nu_truth_dir[1]),
        'nu_dir_truth_z':   float(nu_truth_dir[2]),
        'zenith_truth_deg': zen_t,
        'L_truth_km':       L_t,
        # Reco
        'nu_energy_reco':   nu_e_reco,
        'nu_dir_reco_x':    float(nu_dir_reco[0]),
        'nu_dir_reco_y':    float(nu_dir_reco[1]),
        'nu_dir_reco_z':    float(nu_dir_reco[2]),
        'zenith_reco_deg':  zen_r,
        'L_reco_km':        L_r,
        # Topology
        'n_primaries':       len(primaries),
        'n_total_particles': len(particles),
        'n_primary_leptons': n_lep,
        'n_primary_protons': n_pro,
        'n_primary_pions':   n_pi,
        'n_primary_photons': n_g,
        'n_primary_other':   n_other,
        'leading_lepton_pdg':    leading_lep['pdg'] if leading_lep else None,
        'leading_lepton_energy': leading_lep['energy'] if leading_lep else None,
        # Reco quality
        'n_points_raw':      n_raw,
        'n_points_filtered': n_filt,
        'direction_recon_ok':    direction_ok,
        'is_ccqe_topology':      is_ccqe_topology,
        'is_ccqe_reconstructed': is_ccqe_reco,
    }
    row.update(obs)
    return row

## 5. Smoke test on a few instances

In [5]:
# Quick check on the first few instances before running everything
for i in range(3):
    r = process_instance(i)
    print(f"Instance {i}: nu_flavour={r['nu_flavour']:<6} is_cc={r['is_cc']} "
          f"n_hits={r['n_hits']:<6} n_primaries={r['n_primaries']:<3} "
          f"E_t={r['nu_energy_truth']:.3f} E_r={r['nu_energy_reco']:.3f}")

Instance 0: nu_flavour=nue    is_cc=True n_hits=102    n_primaries=2   E_t=1.015 E_r=0.709
Instance 1: nu_flavour=nue    is_cc=True n_hits=501    n_primaries=3   E_t=0.477 E_r=0.595
Instance 2: nu_flavour=nutau  is_cc=False n_hits=1689   n_primaries=125 E_t=3.716 E_r=67.470


## 6. Run on a sample of instances

Set `N_INSTANCES = None` to process everything (~9,700; takes a while). For a faster demo, leave it at e.g. 3,000.

In [6]:
N_INSTANCES = 1000   # set to None for the full ~9,700 (takes much longer)

n = N_INSTANCES if N_INSTANCES is not None else n_instances
rows = []
for inst in tqdm(range(n), desc='Processing instances'):
    try:
        rows.append(process_instance(inst))
    except Exception as exc:
        print(f'  WARNING instance {inst}: {exc!r}')

df = pd.DataFrame(rows)
print(f'\nBuilt dataset of shape {df.shape}')
print(f'\nLabel breakdown:')
print(f"  CC: {df['is_cc'].sum():>5} ({df['is_cc'].mean():.1%})")
print(f"  NC: {(~df['is_cc']).sum():>5} ({(~df['is_cc']).mean():.1%})")
print(f"\nFlavour breakdown:")
print(df['nu_flavour'].value_counts())

Processing instances:   0%|          | 0/1000 [00:00<?, ?it/s]


Built dataset of shape (1000, 44)

Label breakdown:
  CC:   543 (54.3%)
  NC:   457 (45.7%)

Flavour breakdown:
nu_flavour
nue      443
numu     398
nutau    159
Name: count, dtype: int64


## 7. Save the dataset

Output: `dataset.csv` -- the database used by the EDA and ML notebooks.

In [7]:
df.to_csv('dataset.csv', index=False)
print(f'Wrote dataset.csv  ({len(df):,} rows, {df.shape[1]} columns)')

# ----- Also save as Excel for easy human-readable viewing -----
# Excel opens as a proper aligned grid with headers, filters, frozen rows.
# CSV is just text; .xlsx is the right format if you want to read the data by eye.
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
from openpyxl.styles import Font, PatternFill, Alignment

with pd.ExcelWriter('dataset.xlsx', engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='dataset', index=False, freeze_panes=(1, 2))
    ws = writer.sheets['dataset']
    # Header style: bold white on blue, centred
    header_fill = PatternFill('solid', fgColor='2D6A9F')
    header_font = Font(bold=True, color='FFFFFF')
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal='center', vertical='center')
    # Auto-size column widths to content (capped at 22)
    for i, col in enumerate(df.columns, 1):
        max_len = max(len(str(col)), df[col].astype(str).str.len().max())
        ws.column_dimensions[get_column_letter(i)].width = min(max(max_len + 2, 10), 22)
    # Add an autofilter so columns are sortable / filterable in Excel
    ws.auto_filter.ref = ws.dimensions
print(f'Wrote dataset.xlsx ({len(df):,} rows, {df.shape[1]} columns) -- formatted, sortable')

# ----- Also save as Parquet for fast load in downstream notebooks -----
df.to_parquet('dataset.parquet', index=False)
print(f'Wrote dataset.parquet ({len(df):,} rows, {df.shape[1]} columns) -- compact, fast')

print('\nColumns:')
for c in df.columns: print(f'  {c}')

Wrote dataset.csv  (1,000 rows, 44 columns)


Wrote dataset.xlsx (1,000 rows, 44 columns) -- formatted, sortable
Wrote dataset.parquet (1,000 rows, 44 columns) -- compact, fast

Columns:
  instance_id
  event_id
  is_cc
  nu_pdg
  nu_flavour
  nu_energy_truth
  nu_dir_truth_x
  nu_dir_truth_y
  nu_dir_truth_z
  zenith_truth_deg
  L_truth_km
  nu_energy_reco
  nu_dir_reco_x
  nu_dir_reco_y
  nu_dir_reco_z
  zenith_reco_deg
  L_reco_km
  n_primaries
  n_total_particles
  n_primary_leptons
  n_primary_protons
  n_primary_pions
  n_primary_photons
  n_primary_other
  leading_lepton_pdg
  leading_lepton_energy
  n_points_raw
  n_points_filtered
  direction_recon_ok
  is_ccqe_topology
  is_ccqe_reconstructed
  n_hits
  n_hits_w
  n_hits_v
  n_hits_u
  total_adc
  mean_adc
  max_adc
  drift_range
  drift_std
  channel_range_w
  channel_range_v
  channel_range_u
  hit_density
